# Finetuning Granite Speech on the Synthesized Dataset (LoRA)

In [ ]:
!pip install -U -q "transformers>=4.52" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec peft bitsandbytes

## Setup

In [ ]:
import torch
from pathlib import Path
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

MODEL_ID = "ibm-granite/granite-speech-5.0-2b"

OUTPUT_DIR = Path("checkpoint_synthesized")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_ID, dtype=torch.bfloat16).to("cuda")
print(f"Model loaded on {model.device} with dtype {model.dtype}")

## Load the Synthesized Dataset (Train/Dev Split)

In [ ]:
def read_split(path):
    """Tab-separated manifest: audio path, entity word, transcript."""
    dataset = []
    with open(path, encoding="UTF-8") as rf:
        for line in rf:
            audio, word, text = line.rstrip("\n").split("\t")
            dataset.append({"audio": audio, "word": word, "text": text})
    return dataset

train_data = read_split("synthesized_train.tsv")
dev_data = read_split("synthesized_dev.tsv")

print(len(train_data), "train samples,", len(dev_data), "dev samples")
print(train_data[0])

## Prompt Construction (Mix of With and Without Context)

In [ ]:
import random

random.seed(42)

# context = keyword biasing: about half of the samples get the entity word as a keyword,
# the rest are transcribed without any context
CONTEXT_FRACTION = 0.5

for sample in train_data:
    sample["prompt"] = sample["word"] if random.random() < CONTEXT_FRACTION else None

for sample in dev_data:
    sample["prompt"] = sample["word"] if random.random() < CONTEXT_FRACTION else None

import torchaudio

def load_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)
    waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)
    waveform = waveform.mean(dim=0)
    return waveform

def build_chat_prompt(user_prompt):
    chat = [
        {"role": "system", "content": "You are Granite, developed by IBM. You are a helpful AI assistant."},
        {"role": "user", "content": f"<|audio|>{user_prompt}"},
    ]
    return processor.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

DEFAULT_PROMPT = "can you transcribe the speech into a written format?"

def build_user_prompt(sample):
    prompt = sample.get("prompt")
    if prompt:
        return f"{DEFAULT_PROMPT} Keywords: {prompt}"
    return DEFAULT_PROMPT

with_context = sum(1 for s in train_data if s["prompt"])
print(f"{with_context} of {len(train_data)} train samples with context")

## LoRA Configuration (PEFT)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Collate Function (Audio + Prompt + Target Text)

In [ ]:
def collate_fn(batch):
    wavs = [load_audio(sample["audio"]) for sample in batch]
    prompts = [build_chat_prompt(build_user_prompt(sample)) for sample in batch]

    inputs = processor(prompts, wavs, return_tensors="pt", padding=True)

    eos_id = processor.tokenizer.eos_token_id   # the model must learn to stop
    target_ids = []
    for sample in batch:
        t = processor.tokenizer(sample["text"], add_special_tokens=False,
                                return_tensors="pt").input_ids[0]
        target_ids.append(torch.cat([t, torch.tensor([eos_id], dtype=t.dtype)]))
    batch_size, prompt_len = inputs["input_ids"].shape
    max_len = prompt_len + max(len(t) for t in target_ids)
    pad_id = processor.tokenizer.pad_token_id if processor.tokenizer.pad_token_id is not None else 0

    input_ids = torch.full((batch_size, max_len), pad_id, dtype=inputs["input_ids"].dtype)
    attention_mask = torch.zeros((batch_size, max_len), dtype=inputs["attention_mask"].dtype)
    labels = torch.full((batch_size, max_len), -100, dtype=torch.long)

    input_ids[:, :prompt_len] = inputs["input_ids"]
    attention_mask[:, :prompt_len] = inputs["attention_mask"]
    for i, t in enumerate(target_ids):
        input_ids[i, prompt_len:prompt_len + len(t)] = t
        attention_mask[i, prompt_len:prompt_len + len(t)] = 1
        labels[i, prompt_len:prompt_len + len(t)] = t

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        **{k: v for k, v in inputs.items() if k not in ("input_ids", "attention_mask")},
    }

batch = collate_fn(train_data[:2])
mask = batch["labels"] != -100
print(processor.tokenizer.decode(batch["labels"][0][mask[0]]))   # must end with the EOS token

## Training Configuration

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR.as_posix(),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_train_epochs=3,   # the synthesized dataset is small; more epochs are affordable
    eval_strategy="epoch",
    logging_steps=5,
    logging_first_step=True,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    report_to=[],
    remove_unused_columns=False,
    disable_tqdm=False,
)

## Evaluation (Dev WER on the Synthesized Data)

In [ ]:
import jiwer
from tqdm import tqdm

def compute_dev_wer():
    model.eval()
    wers = []
    for sample in tqdm(dev_data):
        wav = load_audio(sample["audio"])
        text_prompt = build_chat_prompt(build_user_prompt(sample))
        inputs = processor(text_prompt, wav, device=model.device, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=400, do_sample=False, num_beams=1,
                                 use_cache=True)
        hyp = processor.tokenizer.decode(
            out[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
        wers.append(jiwer.wer(sample["text"], hyp))
    model.train()
    return sum(wers) / len(wers)

print("Dev WER before training:", compute_dev_wer())

## Train

In [ ]:
from transformers import Trainer, TrainerCallback


class ConsoleLogCallback(TrainerCallback):
    """Print loss and learning rate at every logging step"""

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"step {state.global_step}: {logs}")


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=dev_data,
    data_collator=collate_fn,
    callbacks=[ConsoleLogCallback()],
)

trainer.train()

In [ ]:
print("Dev WER AFTER training:", compute_dev_wer())

## Write Out the Checkpoint (LoRA Merged into the Base Model)

In [ ]:
merged_model = model.merge_and_unload()   # merge LoRA weights into the base model
merged_model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print(f"Checkpoint written to {OUTPUT_DIR}")